In [ ]:
"""
Step 05: Timestamp Barrier (TSBAR) Target Data Collection

Objective:

    - Replace the step12 TS_delta_matrix_sell target of the previous workspace (legacy) with the rules agreed on 2026-10-01 (trade_execution.py):
        * decision at bar t (feature row) -> entry at the OPEN of bar t+1 of the same session;
        * first entry at the 10:00 open (decision 09:59); the 15:59 row cannot buy at the next session's open;
        * SL / TP computed from the entry open, active from the bar after the entry bar;
        * exit at the bar's open when it opens beyond a level; TP needs one tick through; SL wins ties;
        * maximum 10 trading days (entry day = day 1), time limit exit at the close of the last bar;
        * NA when the data ends before the holding window (regenerate these dates after new data is added).
    - One row per decision bar, one column per delta: "exit_bar_count|exit_price|exit_reason".
    - Costs are NOT stored in the cells; they are applied when the labels are parsed (barrier_labels.add_TSBAR_label_cols),
      so the cost assumptions can change without regenerating this step.
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT PLOTLY EXPRESS
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION
from so import config
# IMPORT TRANSFORM OHLCV DATA FUNCTIONS
from so.features.ohlcv_data_utils import format_ohlcv_pdf
# IMPORT MARKET DATETIME FUNCTIONS
from so.core.datetime_utils import ny_tz, get_date_range_market_schedule_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import get_path_file_list, \
                                    read_pdf_from_csv_file_path_list, \
                                    read_date_range_csv_files_from_path, \
                                    generate_func_data_date_list, \
                                    get_missing_date_list
# IMPORT TRADE EXECUTION FUNCTIONS
from so.core.trade_execution import get_ohlcv_array_dict
# IMPORT BARRIER LABEL FUNCTIONS
from so.features.barrier_labels import get_date_TSBAR_pdf, \
                            get_date_market_open_ts_dict, \
                            add_TSBAR_label_cols, \
                            parse_TSBAR_cell_pdf, \
                            get_delta_col_str, \
                            get_TSBAR_incomplete_date_list, \
                            format_TSBAR_pdf
# IMPORT BACKTEST SIMULATION FUNCTIONS
from so.core.backtest_simulation import simulate_decision_trading_dict
# IMPORT BUY EVALUATION AND PLOT FUNCTIONS
from so.features.buy_eval_utils import plot_date_delta_exit_reason, plot_ideal_pred_buy_data, get_ideal_vs_pred_dict, get_date_TSBAR_trade_pdf
from so.features.plot_ohlcv_utils import plot_transaction_ohlcv
# IMPORT PATHS
from so import paths

In [ ]:
# CALL FUNCTION TO GET ALL FILES IN PATH
rawzone_data_file_path_list = get_path_file_list(paths.LOCAL_OHLCV_DATA_FILE_PATH_STR)[:]
# CALL FUNCTION TO READ PDF FROM PATH LIST
raw_ohlcv_pdf = read_pdf_from_csv_file_path_list(rawzone_data_file_path_list)
# CORRECT THE TIMESTAMP DATE COLUMN
raw_ohlcv_pdf["timestamp"] = pd.to_datetime(raw_ohlcv_pdf.timestamp, utc=True).dt.tz_convert(ny_tz)
raw_ohlcv_pdf["date"] = raw_ohlcv_pdf["timestamp"].dt.date
# CALL FUNCTION TO COLLECT DATA AND SORT BY TIMESTAMP
complete_ohlcv_pdf = format_ohlcv_pdf(raw_ohlcv_pdf.copy()).sort_values("timestamp").reset_index(drop=True)
# DISPLAY INFORMATION
print(f"min timestamp:\t{complete_ohlcv_pdf.timestamp.min()}")
print(f"max timestamp:\t{complete_ohlcv_pdf.timestamp.max()}")
# PREVIEW DATAFRAME
complete_ohlcv_pdf

In [ ]:
# COLLECT THE START AND END DATE
start_date_str, end_date_str = str(complete_ohlcv_pdf["date"].min()), str(complete_ohlcv_pdf["date"].max())
# CALL FUNCTION TO GET THE MARKET SCHEDULE OF THE DATA RANGE
market_schedule_pdf = get_date_range_market_schedule_pdf(start_date_str, end_date_str)
# CREATE THE DATE TO MARKET OPEN DICTIONARY
date_market_open_ts_dict = get_date_market_open_ts_dict(market_schedule_pdf)
# CREATE THE DATE TO (MARKET OPEN, MARKET CLOSE) DICTIONARY (THE CLOSE IS THE LAST BAR TIMESTAMP)
date_market_schedule_dict = dict(zip(pd.to_datetime(market_schedule_pdf["date"]).dt.date,
                                     zip(market_schedule_pdf["market_open_ts"], market_schedule_pdf["market_close_ts"])))
# PREVIEW DATAFRAME
market_schedule_pdf

In [ ]:
# CALL FUNCTION TO CONVERT THE COMPLETE DATA TO ARRAYS (MULTI-DAY TRADES NEED THE FOLLOWING SESSIONS)
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
# DISPLAY INFORMATION
print(f"Bar Count:\t{len(ohlcv_array_dict['close_arr']):,}")
print(f"Session Count:\t{len(ohlcv_array_dict['session_date_list']):,}")
print(f"Delta Count:\t{len(config.FULL_DELTA_LIST)}\t(RR ratio: {config.RR_RATIO}, max hold: {config.MAX_HOLD_TRADING_DAYS} trading days)")

In [ ]:
# FUNCTION: GENERATE THE DATE TIMESTAMP BARRIER (TSBAR) DATAFRAME
def get_date_TSBAR_data_pdf(date_str_in):
    # CALL FUNCTION TO GENERATE THE DATE TSBAR DATAFRAME
    return get_date_TSBAR_pdf(ohlcv_array_dict, date_market_open_ts_dict, date_str_in, config.FULL_DELTA_LIST, config.RR_RATIO)

# DEFINE SAMPLE DATE (A SESSION WITH A COMPLETE HOLDING WINDOW)
sample_date_str = str(ohlcv_array_dict["session_date_list"][-30])
# CALL FUNCTION TO GENERATE THE SAMPLE DATE TSBAR DATAFRAME
sample_date_TSBAR_pdf = get_date_TSBAR_data_pdf(sample_date_str)
# PREVIEW DATAFRAME
sample_date_TSBAR_pdf

In [ ]:
# CALL FUNCTION TO PARSE THE LABELS OF THE SAMPLE DATE
sample_label_pdf = add_TSBAR_label_cols(sample_date_TSBAR_pdf, config.FULL_DELTA_LIST, drop_cell_cols_in=True)
# LIST TO HOLD THE DELTA SUMMARIES
delta_summary_dict_list = []
# ITERATE OVER THE DELTAS
for delta in config.FULL_DELTA_LIST:
    # DEFINE THE DELTA COLUMN
    delta_col_str = get_delta_col_str(delta)
    # COLLECT THE EXIT REASON SHARES
    reason_share_dict = sample_label_pdf[f"exit_reason_{delta_col_str}"].value_counts(normalize=True).to_dict()
    # APPEND THE SUMMARY
    delta_summary_dict_list.append({"delta": delta, **{f"{reason}_share": reason_share_dict.get(reason, 0.0) for reason in ["TP", "SL", "TL", "NA"]},
                                    "mean_net_return": sample_label_pdf[f"net_return_{delta_col_str}"].mean(),
                                    "median_exit_bar_count": sample_label_pdf[f"exit_bar_count_{delta_col_str}"].median()})
# PREVIEW DATAFRAME
pd.DataFrame(delta_summary_dict_list)

In [ ]:
"""
Consistency check: a simulated buy at a decision row must exit exactly like the TSBAR cell of that row
(same exit reason, same raw exit price, same number of bars). Both use trade_execution.resolve_entry_trade_dict.
"""
# CREATE THE RANDOM GENERATOR
rng = np.random.default_rng(config.RANDOM_SEED)
# DEFINE THE MISMATCH LIST
mismatch_list = []
# ITERATE OVER RANDOM DECISION ROWS AND DELTAS
for row_idx in rng.choice(len(sample_date_TSBAR_pdf), size=20, replace=False):
    for delta in rng.choice(config.FULL_DELTA_LIST, size=3, replace=False):
        # SIMULATE A BUY AT THE DECISION ROW
        decision_pdf = pd.DataFrame({"decision_ts": [sample_date_TSBAR_pdf["decision_ts"].iloc[row_idx]], "buy_flag": [True], "delta": [delta]})
        transaction_row = simulate_decision_trading_dict(decision_pdf, ohlcv_array_dict)["transaction_pdf"].iloc[0]
        # PARSE THE TSBAR CELL
        cell_row = parse_TSBAR_cell_pdf(sample_date_TSBAR_pdf[get_delta_col_str(delta)].iloc[[row_idx]]).iloc[0]
        # IF THE EXITS DIFFER
        if (transaction_row["exit_reason"] != cell_row["exit_reason"]) or (not np.isclose(transaction_row["sell_raw_price"], cell_row["exit_price"])) or (transaction_row["hold_bar_count"] != cell_row["exit_bar_count"]):
            # ADD THE MISMATCH
            mismatch_list.append((row_idx, delta))
# DISPLAY INFORMATION
print("✅ TSBAR cells match the simulator" if not mismatch_list else f"❌ Mismatches: {mismatch_list}")

In [ ]:
# CALL FUNCTION TO PLOT THE SAMPLE DATE EXIT REASONS (buy_eval_utils)
plot_date_delta_exit_reason(sample_label_pdf, config.FULL_DELTA_LIST, f"Exit Reason Per Delta For '{sample_date_str}'")

In [ ]:
# DEFINE THE PREVIEW DELTA
preview_delta = 0.002
# DEFINE A SAMPLE "PRED" DECISION DICTIONARY (EVERY 30 MINUTES FROM 09:59, FOR ILLUSTRATION ONLY)
sample_ts_BS_dict = {ts: 1 for ts in sample_date_TSBAR_pdf["decision_ts"] if (ts.hour * 60 + ts.minute - 599) % 30 == 0}
# CALL FUNCTION TO PLOT THE IDEAL (TAKE PROFIT) ENTRIES AND THE SAMPLE DECISIONS WITH THEIR SLTP BOUNDARIES
plot_ideal_pred_buy_data(complete_ohlcv_pdf, sample_date_TSBAR_pdf, ohlcv_array_dict, preview_delta, sample_date_str, sample_ts_BS_dict)
# CALL FUNCTION TO COMPARE THE SAMPLE DECISIONS WITH THE IDEAL ENTRIES
get_ideal_vs_pred_dict(get_date_TSBAR_trade_pdf(sample_date_TSBAR_pdf, preview_delta, ohlcv_array_dict), sample_ts_BS_dict)

In [ ]:
# SIMULATE THE SAMPLE DECISIONS WITH THE BACKTEST SIMULATOR (ONE POSITION AT A TIME, MULTI-DAY HOLDING)
sample_decision_pdf = pd.DataFrame({"decision_ts": list(sample_ts_BS_dict.keys()), "buy_flag": True, "delta": preview_delta})
sample_transaction_pdf = simulate_decision_trading_dict(sample_decision_pdf, ohlcv_array_dict)["transaction_pdf"]
# CALL FUNCTION TO PLOT THE SIMULATED TRADES (THE CANDLES COVER THE SAMPLE DATE AND THE DAYS UNTIL THE LAST EXIT)
plot_transaction_ohlcv(complete_ohlcv_pdf[(complete_ohlcv_pdf["timestamp"] >= sample_transaction_pdf["buy_ts"].min() - pd.Timedelta(minutes=30))
                                          & (complete_ohlcv_pdf["timestamp"] <= sample_transaction_pdf["sell_ts"].max() + pd.Timedelta(minutes=30))],
                       sample_transaction_pdf, f"{sample_date_str} (delta {preview_delta:.2%})") if not sample_transaction_pdf.empty else None
# PREVIEW DATAFRAME
sample_transaction_pdf

In [ ]:
# FIND THE DATES WHOSE HOLDING WINDOWS WERE INCOMPLETE WHEN GENERATED (REGENERATE AFTER NEW DATA IS ADDED)
incomplete_date_list = get_TSBAR_incomplete_date_list(paths.LOCAL_TSBAR_DATA_FILE_PATH_STR)
# DISPLAY INFORMATION
print(f"Incomplete Date Count:\t{len(incomplete_date_list)}")
# PREVIEW LIST
incomplete_date_list

In [ ]:
# CALL FUNCTION TO REGENERATE THE INCOMPLETE DATES (OVERWRITE MODE)
generate_func_data_date_list(get_date_TSBAR_data_pdf, incomplete_date_list, paths.LOCAL_TSBAR_DATA_FILE_PATH_STR, "TSBAR_data", "W")

In [ ]:
# GET MISSING DATE LIST
missing_date_list = get_missing_date_list(paths.LOCAL_TSBAR_DATA_FILE_PATH_STR, start_date_str, end_date_str)
# PREVIEW LIST
missing_date_list[:10]

In [ ]:
# CALL FUNCTION TO PROCESS DATE LIST
generate_func_data_date_list(get_date_TSBAR_data_pdf, missing_date_list, paths.LOCAL_TSBAR_DATA_FILE_PATH_STR, "TSBAR_data", "I")

In [ ]:
# CALL FUNCTION TO READ THE LAST MONTH OF TSBAR DATA
TSBAR_data_pdf = format_TSBAR_pdf(read_date_range_csv_files_from_path(paths.LOCAL_TSBAR_DATA_FILE_PATH_STR, str((pd.to_datetime(end_date_str) - pd.DateOffset(months=1)).date()), end_date_str))
# DISPLAY INFORMATION
print(f"Rows With Incomplete Holding Windows:\t{(~TSBAR_data_pdf['horizon_complete'].astype(bool)).sum():,}")
# PREVIEW DATAFRAME
TSBAR_data_pdf